In [1]:
from pathlib import Path
import os
import json
from datetime import datetime

In [2]:
from main_train import get_args
from utils.config import load_config
from data.dataset import get_vqav2
from data.dataset import get_filtered_trainval
from data.text_processing import save_tokenizer

from data.custom_generators import get_custom_generators
from models.vqa_models import get_model

from train.train import KD_train_model

from data.onehot_encoder import OneHotEncoder

2025-08-06 16:43:10.291620: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754491390.313994   17455 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754491390.320974   17455 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-06 16:43:10.345500: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
config = load_config()
config['model']

{'model_architecture': 'MFBBaseline',
 'max_length': 15,
 'num_vocab_words': -1,
 'min_frequency': 5,
 'image_size': 224,
 'num_channels': 3,
 'num_classes': 1000,
 'consider_teacher': True,
 'k_window': 5,
 'output_MFB': 1024,
 'num_attention_glimps': 2,
 'embedding_dim': 100,
 'use_glove': True,
 'dropout_rate': 0}

In [4]:
config['training']

{'num_epochs': 10,
 'lr': 0.0001,
 'batch_size': 32,
 'alpha': 0.1,
 'temperature': 3,
 'knowledge_distillation': True,
 'restore_best_weights': False}

In [5]:
now = datetime.now().strftime("%y%m%d_%H%M")
saving_folder = config["paths"]["output_path"] / f"{'MFBBaseline'}_{now}"
saving_folder.mkdir(parents=True, exist_ok=True)

config["paths"]["saving_folder"] = saving_folder
config["model"]["model_architecture"] = "MFBBaseline"
# config["training"]["knowledge_distillation"] = args.distill
config["training"]["knowledge_distillation"] = True

In [6]:
saving_folder

PosixPath('outputs/MFBBaseline_250806_1643')

In [7]:
df_train, df_val = get_filtered_trainval(
    config, consider_teacher=config["model"]["consider_teacher"], verbose=True
)

Set:  train2014
Total number of sample in train2014: 443757
Number of training samples after filtering: 388273 ( 87.50 % )
Set:  val2014
Total number of sample in val2014: 214354
Number of validation samples after filtering: 186549 ( 87.03 % )


In [8]:
len(set(df_train["normalized_answer"]))

1000

In [9]:
if config["model"]["num_vocab_words"] > 0:
    config["model"]["min_frequency"] = 0
    num_words = config["model"]["num_vocab_words"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index{num_words}.json"
else:
    mf = config["model"]["min_frequency"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index_mf{mf}.json"
if not tokenizer_path.is_file():
    #save_tokenizer(config, tokenizer_path, verbose=False)
    pass

In [10]:
tokenizer_path 

PosixPath('outputs/word_index_mf5.json')

In [11]:
ct = 'ct' if config["model"]["consider_teacher"] else ''
num_classes = config["model"]["num_classes"]
possible_ans_path = config["paths"]["output_path"] / f"possible_answers_{ct}{num_classes}.json"
possible_ans_path

PosixPath('outputs/possible_answers_ct1000.json')

In [12]:
if not possible_ans_path.is_file():
    enc = OneHotEncoder()
    train_ans = list(df_train["normalized_answer"])
    enc.fit(train_ans)
    enc.save_json(possible_ans_path)

In [13]:
config["training"]["knowledge_distillation"] = True

In [14]:
train_data, valid_data = get_custom_generators(
    df_train, df_val, tokenizer_path, possible_ans_path, config
)

In [29]:
inp, outp, w = train_data[0]
q, im = inp
gt, logits = outp
q.shape, im.shape, gt.shape, logits.shape, w.shape

((32, 15), (32, 224, 224, 3), (32, 1000), (32, 1000), (32,))

In [30]:
gt.argmax(axis=-1)

array([ 17, 608,   1, 639, 607, 563, 995, 995, 697, 793,  17, 800, 393,
       563, 995, 563, 998, 799, 406,  98, 563, 563, 527, 998, 995, 990,
       107, 995, 563, 995, 563, 563])

In [31]:
logits.argmax(axis=-1)

array([ 31, 340, 563, 639, 607, 563, 995, 995, 697, 793,  17, 800, 393,
       563, 563, 563, 998, 799, 124,  98, 563, 563, 527, 998, 995, 990,
       107, 995, 563, 995, 563, 563])

In [15]:
config['model']['model_architecture']

'MFBBaseline'

In [16]:
model = get_model(config, tokenizer_path)

I0000 00:00:1754475810.543329   12464 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5169 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1060 6GB, pci bus id: 0000:21:00.0, compute capability: 6.1


Number of parameters: 24419360


In [17]:
config['training']['num_epochs'] = 2
config['training']['num_epochs']

2

In [18]:
model = KD_train_model(model, train_data, valid_data, config)

Epoch 1/2


I0000 00:00:1754475824.686314   12578 service.cc:148] XLA service 0x742c8c04d9f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754475824.686608   12578 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce GTX 1060 6GB, Compute Capability 6.1
2025-08-06 12:23:44.998759: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1754475826.397309   12578 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-08-06 12:23:52.287033: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


    2/12134 ━━━━━━━━━━━━━━━━━━━━ 13:20 66ms/step - accuracy: 0.0000e+00 - distillation_loss: 0.0740 - loss: 0.0690 - student_loss: 0.0239  

I0000 00:00:1754475836.046219   12578 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12134/12134 ━━━━━━━━━━━━━━━━━━━━ 4214s 346ms/step - accuracy: 0.3553 - distillation_loss: 0.0135 - loss: 0.0129 - student_loss: 0.0075 - val_distillation_loss: 0.0088 - val_loss: 0.0084 - val_student_loss: 0.0053 - learning_rate: 1.0000e-04
Epoch 2/2
12134/12134 ━━━━━━━━━━━━━━━━━━━━ 4248s 350ms/step - accuracy: 0.4807 - distillation_loss: 0.0083 - loss: 0.0080 - student_loss: 0.0048 - val_distillation_loss: 0.0080 - val_loss: 0.0077 - val_student_loss: 0.0050 - learning_rate: 1.0000e-04


In [20]:
arch = config["model"]["model_architecture"]
model.save(config['paths']['saving_folder'] / f"trained_{arch}.keras")